In [ ]:
import torch
device = torch.device(
"cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

Device: cuda


In [ ]:
sentences = [
"king is a royal man",
"queen is a royal woman",
"king rules the kingdom",
"queen rules the kingdom",
"king lives in the palace",
"queen lives in the palace",
"prince is a young man",
"princess is a young woman",
"prince lives in the palace",
"princess lives in the palace",
"man is a male person",
"woman is a female person",
"boy is a young male",
"girl is a young female",
"dog is an animal",
"cat is an animal",
"dog is a pet",
"cat is a pet",
"puppy is a young dog",
"kitten is a young cat",
"apple is a fruit",
"banana is a fruit",
"orange is a fruit",
"apple is sweet",
"banana is sweet",
"orange is sweet",
"car is a vehicle",
"bus is a vehicle",
"truck is a vehicle",
"car moves on road",
"bus moves on road",
"truck moves on road",
]


# Tokenize - 토큰화
corpus = []

for sentence in sentences:
    tokens = sentence.lower().split()
    corpus.append(tokens)

for tokens in corpus[:5]:
    print(tokens)


# Build vocabulary - 단어장 40개 생성 -> 40개의 고유 단어를 모아서 각각 ID 부여
words = []
for sentence in corpus:
   words.extend(sentence)

# 단어장 생성
vocab = sorted(set(words))

# 단어에 ID 부여
word_to_id = {
    word: i
    for i, word in enumerate(vocab)
}

id_to_word = {
    i: word
    for word, i in word_to_id.items()
}
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)
print(word_to_id)

# Generate Skip-gram Training Pair
# corpus 순회하면서 중심, 주변 학습 데이터 쌍 생성 / window 크기 2로 설정
def generate_skipgram_pairs(corpus, window=2):
    pairs = []

    for sentence in corpus:

        for i in range(len(sentence)):
            center_word = sentence[i] # 중심 단어
            start = max(0, i - window)
            end = min(len(sentence),i + window + 1)

            for j in range(start, end):
                if i == j:
                    continue

                context_word = sentence[j]

                center_id = word_to_id[center_word]
                context_id = word_to_id[context_word]
                pairs.append((center_id, context_id))
    return pairs

['king', 'is', 'a', 'royal', 'man']
['queen', 'is', 'a', 'royal', 'woman']
['king', 'rules', 'the', 'kingdom']
['queen', 'rules', 'the', 'kingdom']
['king', 'lives', 'in', 'the', 'palace']
Vocabulary size: 40
{'a': 0, 'an': 1, 'animal': 2, 'apple': 3, 'banana': 4, 'boy': 5, 'bus': 6, 'car': 7, 'cat': 8, 'dog': 9, 'female': 10, 'fruit': 11, 'girl': 12, 'in': 13, 'is': 14, 'king': 15, 'kingdom': 16, 'kitten': 17, 'lives': 18, 'male': 19, 'man': 20, 'moves': 21, 'on': 22, 'orange': 23, 'palace': 24, 'person': 25, 'pet': 26, 'prince': 27, 'princess': 28, 'puppy': 29, 'queen': 30, 'road': 31, 'royal': 32, 'rules': 33, 'sweet': 34, 'the': 35, 'truck': 36, 'vehicle': 37, 'woman': 38, 'young': 39}


1. Train the model with two different window sizes

• window = 1


• window = 4


Then compare the top-5 similar words for: words = ["king", "queen",
"dog", "apple", "car"]
Briefly explain how the context window changes the results.

In [ ]:
pairs1 = generate_skipgram_pairs(corpus,window=1)
pairs4 = generate_skipgram_pairs(corpus,window=4)

print("Number of training pairs1:", len(pairs1))
print("Number of training pairs4:", len(pairs4))

# 페어 출력해서 보기!
print("pair1")
for center_id, context_id in pairs1[:20]:
    print(id_to_word[center_id], "->", id_to_word[context_id])

print("\n\npair4")
for center_id, context_id in pairs4[:20]:
    print(id_to_word[center_id], "->", id_to_word[context_id])

Number of training pairs1: 214
Number of training pairs4: 478
pair1
king -> is
is -> king
is -> a
a -> is
a -> royal
royal -> a
royal -> man
man -> royal
queen -> is
is -> queen
is -> a
a -> is
a -> royal
royal -> a
royal -> woman
woman -> royal
king -> rules
rules -> king
rules -> the
the -> rules


pair4
king -> is
king -> a
king -> royal
king -> man
is -> king
is -> a
is -> royal
is -> man
a -> king
a -> is
a -> royal
a -> man
royal -> king
royal -> is
royal -> a
royal -> man
man -> king
man -> is
man -> a
man -> royal


In [ ]:
# win = 1
center_words1 = torch.tensor(
    [pair[0] for pair in pairs1],
    dtype=torch.long
)

context_words1 = torch.tensor(
    [pair[1] for pair in pairs1],
    dtype=torch.long
)

print(center_words1.shape)
print(context_words1.shape)


# win = 4
center_words4 = torch.tensor(
    [pair[0] for pair in pairs4],
    dtype=torch.long
)

context_words4 = torch.tensor(
    [pair[1] for pair in pairs4],
    dtype=torch.long
)

print(center_words4.shape)
print(context_words4.shape)


torch.Size([214])
torch.Size([214])
torch.Size([478])
torch.Size([478])
torch.Size([214])


In [ ]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

# win = 1
# 중심, 주변으로 묶기
dataset1 = TensorDataset(center_words1,context_words1)

# 64개의 배치 사이즈로 제공
loader1 = DataLoader(dataset1,batch_size=64,shuffle=True)

# win = 4
# 중심, 주변으로 묶기
dataset4 = TensorDataset(center_words4,context_words4)

# 64개의 배치 사이즈로 제공
loader4 = DataLoader(dataset4,batch_size=64,shuffle=True)

In [ ]:
# Skip-gram 신경망 생성
import torch.nn as nn

class SkipGramModel(nn.Module):
    def __init__( self, vocab_size, embedding_dim ):

        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, center_words):

        # 중심 단어 ID를 embedding vector로 바꾸기
        embedded = self.embedding(center_words)
        # 밑LLM에서 배운 것 처럼 특정 단어를 통과 시키면 해당 단어에 대한 embedding_dim차원 백터가 나옴!

        # embedding(백터를) -> vocabulary logits(40개의 점수 결과가 나온당)
        logits = self.output(embedded)

        return logits

In [ ]:
# win 1을 위한 model 생성
embedding_dim = 50
model1 = SkipGramModel(vocab_size,embedding_dim)
model1 = model1.to(device)
print(model1)

# win 4을 위한 model 생성
embedding_dim = 50
model4 = SkipGramModel(vocab_size,embedding_dim)
model4 = model4.to(device)
print(model4)

NameError: name 'vocab_size' is not defined

In [ ]:
# Loss & optimizer
criterion = nn.CrossEntropyLoss()

optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.01)  # model1 전용
optimizer4 = torch.optim.Adam(model4.parameters(), lr=0.01)  # model4 전용

In [ ]:
# Word2Vec 훈련

# 200번
epochs = 200

print("window = 1")
for epoch in range(epochs):
    total_loss = 0

    for centers, contexts in loader1:
        centers = centers.to(device)
        contexts = contexts.to(device)

        # 그레디언트 초기화
        optimizer1.zero_grad()

        logits = model1(centers)

        loss = criterion(logits,contexts)

        loss.backward()

        optimizer1.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        average_loss = (total_loss / len(loader1))
        print(
            f"Epoch {epoch+1:3d}, "
            f"Loss: {average_loss:.4f}")

print("window = 4")
for epoch in range(epochs):
    total_loss = 0

    for centers, contexts in loader4:
        centers = centers.to(device)
        contexts = contexts.to(device)

        # 그레디언트 초기화
        optimizer4.zero_grad()

        logits = model4(centers)

        loss = criterion(logits,contexts)

        loss.backward()

        optimizer4.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        average_loss = (total_loss / len(loader4))
        print(
            f"Epoch {epoch+1:3d}, "
            f"Loss: {average_loss:.4f}")

In [ ]:
print(1)
embeddings1 = model1.embedding.weight
print(embeddings1.shape)
print(embeddings1.device)

print(4)
embeddings4 = model4.embedding.weight
print(embeddings4.shape)
print(embeddings4.device)

king_id = word_to_id["king"]
king_vector1 = embeddings1[king_id]
king_vector4 = embeddings4[king_id]

# king에 해당하는 임베딩 백터 출력
print(king_vector1)
print(king_vector4)

In [ ]:
import torch.nn.functional as F

# 가장 비슷한 단어 top-k 찾기
def most_similar(embeddings, word, top_k=5):
    # 백터의 크기를 1로 만들기
    normalized = F.normalize(embeddings, p=2, dim=1)
    target = normalized[word_to_id[word]]

    # 단어장의 단어 40개와 COS 유사도를 한 번에 계산하기
    similarities = torch.matmul(normalized, target)

    # 유사도 큰 순
    values, indices = torch.topk(similarities, k=top_k + 1)

    results = []
    for score, idx in zip(values.tolist(), indices.tolist()):
        candidate = id_to_word[idx]

        if candidate != word:
            results.append((candidate, round(score, 3)))

        if len(results) == top_k:
            break
    return results

# window 1 vs window 4 비교하기
words = ["king", "queen", "dog", "apple", "car"]
for w in words:
    print(f"[{w}]")
    print("  window=1:", most_similar(embeddings1, w))
    print("  window=4:", most_similar(embeddings4, w))
    print()